# Title/Abstract Screening with AgenticWorkflow

This tutorial demonstrates **agentic title/abstract screening** — where the reviewer actively searches the web, builds memory, and flags uncertain items during the review process.

We will:
1. Load a real dataset of 978 medical AI articles and select a mixed subset of 10
2. Screen articles with a `TitleAbstractReviewer` that uses **DuckDuckGo search**, **memory**, and **flagging** skills
3. The inclusion criteria intentionally require the agent to **search** for information not in the abstract (journal impact, external validation status)
4. Inspect **action logs** showing tool usage per item (iterations, tool calls, tokens)
5. Inspect **memory** the agent accumulated and **flags** it raised

**Requirements:** `OPENAI_API_KEY` set in your environment or `.env` file.

In [1]:
from dotenv import load_dotenv
load_dotenv()

import json
import os
import pandas as pd
from pathlib import Path
from lattereview.agentic import TitleAbstractReviewer, AgenticWorkflow

## Load and Prepare the Dataset

The dataset has 978 radiology/AI articles. We select 10 articles — 5 about cardiovascular imaging and 5 about non-cardiovascular topics (lung, brain, etc.) — to create a realistic mix of includes and excludes.

In [2]:
df_full = pd.read_csv("data.csv")
print(f"Full dataset: {len(df_full)} articles")

# Select 5 cardiovascular + 5 non-cardiovascular articles for a realistic mix
cardio = df_full[df_full["cardiovascular"] == 1].head(5)
non_cardio = df_full[df_full["cardiovascular"] == 0].head(5)
df = pd.concat([cardio, non_cardio]).reset_index(drop=True)

print(f"\nSelected {len(df)} articles for screening:")
for i, row in df.iterrows():
    label = "CARDIO" if row["cardiovascular"] == 1 else "OTHER"
    print(f"  [{label}] {row['title'][:80]}...")

Full dataset: 978 articles

Selected 10 articles for screening:
  [CARDIO] Automated identification of pulmonary arteries and veins depicted in non-contras...
  [CARDIO] Automatic Calcium Scoring in Low-Dose Chest CT Using Deep Neural Networks With D...
  [CARDIO] Cardiac Rhythm Device Identification Using Neural Networks...
  [CARDIO] Cardiothoracic ratio measurement using artificial intelligence: observer and met...
  [CARDIO] A semi-automatic approach for epicardial adipose tissue segmentation and quantif...
  [OTHER] (18)F-FDG PET/CT Uptake Classification in Lymphoma and Lung Cancer by Using Deep...
  [OTHER] (18)F-FDG-PET/CT Whole-Body Imaging Lung Tumor Diagnostic Model: An Ensemble E-R...
  [OTHER] 3-D Convolutional Neural Networks for Automatic Detection of Pulmonary Nodules i...
  [OTHER] 3D CNN with Visual Insights for Early Detection of Lung Cancer Using Gradient-We...
  [OTHER] 3D deep learning based classification of pulmonary ground glass opacity nodules ...


## Create the Agentic Screener

The key difference from v1: this reviewer has **skills** that let it go beyond what's in the title and abstract.

The inclusion criteria intentionally require information the agent must **search for**:
- Whether the study uses deep learning (sometimes ambiguous from abstracts alone)
- Whether the study was published in a reputable radiology or cardiology journal
- Whether external validation was performed

The agent uses:
- `searching-duckduckgo` — to look up articles and verify claims
- `managing-memory` — to remember patterns across articles (auto-included in agentic mode)
- `flagging-items` — to flag uncertain cases for human review (auto-included in agentic mode)

> **Note:** `managing-memory` and `flagging-items` are automatically included whenever `max_iterations > 1`. You only need to explicitly list additional skills like `searching-duckduckgo`.

In [ ]:
screener = TitleAbstractReviewer(
    name="Screener",
    backstory=(
        "You are a systematic review expert screening articles for a review on "
        "deep learning applications in cardiovascular imaging. You actively search "
        "for articles online when you need to verify claims, check journal quality, "
        "or determine if external validation was performed. You save useful patterns "
        "to memory and flag borderline cases."
    ),
    model="openai:gpt-5.4-mini",
    inclusion_criteria=(
        "Include studies that: (1) apply deep learning (CNN, transformer, or similar) "
        "to cardiovascular imaging (cardiac CT, cardiac MRI, echocardiography, "
        "coronary angiography, cardiac PET), AND (2) were published in a reputable "
        "peer-reviewed journal. Search online if you are unsure about the journal "
        "or whether the study truly uses deep learning for cardiovascular imaging."
    ),
    exclusion_criteria=(
        "Exclude: (1) studies on non-cardiovascular organs (lung, brain, liver, etc.), "
        "(2) studies using only traditional ML (random forest, SVM) without deep learning, "
        "(3) review articles, editorials, or commentaries without original data, "
        "(4) studies focused only on device identification without diagnostic imaging."
    ),
    max_iterations=15,          # Enough room for search + reasoning
    agentic_effort="high",      # Encourage active tool use
    skills=["searching-duckduckgo"],  # managing-memory and flagging-items are auto-included
)

print(f"Reviewer: {screener.name}")
print(f"Model: {screener.model}")
print(f"Skills: {screener.skills}")
print(f"Max iterations: {screener.max_iterations}")
print(f"Agentic effort: {screener.agentic_effort}")

## Run the Agentic Workflow

We run the screener on all 10 articles via `AgenticWorkflow`. The `working_dir` persists memory, flags, logs, and results.

In [4]:
import shutil

WORKING_DIR = Path("./screening_output")
if WORKING_DIR.exists():
    shutil.rmtree(WORKING_DIR)

workflow = AgenticWorkflow(
    workflow_schema=[
        {
            "round": "A",
            "reviewers": [screener],
            "text_inputs": ["title", "abstract"],
        }
    ],
    working_dir=WORKING_DIR,
    verbose=True,
)

result_df = await workflow(df)

print(f"\nProcessed {len(result_df)} articles")
print(f"Total cost: ${workflow.total_cost:.4f}")


====== Starting review round A (1/1) ======

Processing 10 eligible rows
Running reviewer: Screener (10 items)


/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


...

/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Reviewer Screener: 10 successful, 0 failed
Columns after Screener: ['title', 'abstract', 'DOI', 'study_type', 'clinical_application', 'organ', 'modality', 'cardiovascular', 'task', 'deep_learning', 'external_validation', 'round-A_Screener_output', 'round-A_Screener_reasoning', 'round-A_Screener_decision', 'round-A_Screener_certainty']
Saved snapshot after round A

Workflow complete. Total cost: $0.0000

Processed 10 articles
Total cost: $0.0000


## Review Results

The workflow adds columns: `round-A_Screener_decision`, `round-A_Screener_reasoning`, `round-A_Screener_certainty`.

In [5]:
print(f"{'Decision':<12} {'Cert':>4} {'Ground Truth':>12}  Title")
print("-" * 100)
for idx, row in result_df.iterrows():
    decision = str(row.get("round-A_Screener_decision", "N/A"))[:10]
    certainty = row.get("round-A_Screener_certainty", "?")
    truth = "CARDIO" if row["cardiovascular"] == 1 else "OTHER"
    title = row["title"][:65]
    print(f"{decision:<12} {certainty:>4}  {truth:>12}  {title}...")

Decision     Cert Ground Truth  Title
----------------------------------------------------------------------------------------------------
exclude        96        CARDIO  Automated identification of pulmonary arteries and veins depicted...
include        93        CARDIO  Automatic Calcium Scoring in Low-Dose Chest CT Using Deep Neural ...
1              97        CARDIO  Cardiac Rhythm Device Identification Using Neural Networks...
5              88        CARDIO  Cardiothoracic ratio measurement using artificial intelligence: o...
exclude        98        CARDIO  A semi-automatic approach for epicardial adipose tissue segmentat...
exclude        99         OTHER  (18)F-FDG PET/CT Uptake Classification in Lymphoma and Lung Cance...
exclude        99         OTHER  (18)F-FDG-PET/CT Whole-Body Imaging Lung Tumor Diagnostic Model: ...
exclude       100         OTHER  3-D Convolutional Neural Networks for Automatic Detection of Pulm...
exclude        99         OTHER  3D CNN with Visual 

## Inspect Action Logs — Tool Usage per Item

The working directory contains JSONL action logs for each item. These logs show:
- How many LLM requests (iterations) each item required
- How many tool calls were made (search, memory, flagging)
- Total tokens used

This gives you visibility into how the agentic loop actually worked.

In [6]:
# Read action logs for each item
logs_dir = WORKING_DIR / "round_A" / "agent_Screener" / "logs"

if logs_dir.exists():
    print(f"{'Item':<10} {'Requests':>9} {'Tool Calls':>11} {'Tokens':>10}")
    print("-" * 45)
    
    for log_file in sorted(logs_dir.glob("*.jsonl")):
        entries = []
        for line in log_file.read_text().strip().split("\n"):
            if line:
                entries.append(json.loads(line))
        
        item_id = log_file.stem.replace("item_", "")
        
        # Find the review_complete entry for stats
        for entry in entries:
            if entry.get("action_type") == "review_complete":
                details = entry.get("details", {})
                requests = details.get("requests", "?")
                tool_calls = details.get("tool_calls", "?")
                tokens = details.get("total_tokens", "?")
                print(f"{item_id:<10} {requests:>9} {tool_calls:>11} {tokens:>10}")
else:
    print("No logs directory found.")

Item        Requests  Tool Calls     Tokens
---------------------------------------------
A-0                4           3       5270
A-1                5           4       7638
A-2                5           4       7648
A-3                5           4       8542
A-4                4           3       5736
A-5                2           1       2168
A-6                2           1       1798
A-7                2           1       1767
A-8                2           1       1713
A-9                2           1       1707


## Inspect Agent Memory

The agent accumulated memories as it screened articles. These carry forward across items — patterns learned from early articles inform later decisions.

In [7]:
memory_dir = WORKING_DIR / "round_A" / "agent_Screener" / "memory"
index_file = memory_dir / "_index.json"

if index_file.exists():
    with open(index_file) as f:
        raw = json.load(f)
    memories = raw.get("memories", raw) if isinstance(raw, dict) else raw
    print(f"Agent saved {len(memories)} memories:\n")
    for mem in memories:
        mem_id = mem.get("id", "?")
        brief = mem.get("brief", mem.get("title", ""))
        print(f"  [{mem_id}] {brief}")
        
        # Read full memory content
        mem_file = memory_dir / f"{mem_id}.md"
        if mem_file.exists():
            content = mem_file.read_text().strip()
            print(f"    Content: {content[:300]}")
        print()
else:
    print("No memories saved.")

No memories saved.


## Inspect Flagged Items

Items the agent flagged for human review — cases where it was uncertain about inclusion/exclusion.

In [8]:
flags_file = WORKING_DIR / "round_A" / "agent_Screener" / "flags" / "flags.json"

if flags_file.exists():
    with open(flags_file) as f:
        flags = json.load(f)
    if flags:
        print(f"Agent flagged {len(flags)} items for human review:\n")
        for flag in flags:
            resolved = "resolved" if flag.get("resolved") else "NEEDS REVIEW"
            print(f"  [{resolved}] Item {flag.get('item_id', '?')}")
            print(f"    Reason: {flag.get('reason', '')}")
            print()
    else:
        print("No items flagged.")
else:
    print("No flags file found.")

No flags file found.


## Inspect a Single Item's Reasoning

Let's look at the full reasoning for one article to see how the agent arrived at its decision.

In [9]:
# Show full reasoning for the first article
row = result_df.iloc[0]
print(f"Title: {row['title']}\n")
print(f"Decision: {row.get('round-A_Screener_decision', 'N/A')}")
print(f"Certainty: {row.get('round-A_Screener_certainty', 'N/A')}")
print(f"\nFull Reasoning:\n{row.get('round-A_Screener_reasoning', 'N/A')}")

Title: Automated identification of pulmonary arteries and veins depicted in non-contrast chest CT scans

Decision: exclude
Certainty: 96

Full Reasoning:
The study uses a convolutional neural network, so it does involve deep learning. However, the imaging target is pulmonary arteries and veins in chest CT, which is fundamentally a lung/thoracic imaging task rather than a cardiovascular imaging study as defined in the review criteria. The exclusion criteria explicitly exclude non-cardiovascular organs such as lung, and this paper’s primary focus is vessel identification within the lungs on non-contrast chest CT. I also do not need to verify journal quality to make the main screening decision because the organ scope alone is disqualifying. Therefore, the article should be excluded.


## Working Directory Structure

The full directory tree shows everything the workflow persisted.

In [10]:
if WORKING_DIR.exists():
    for root, dirs, files in os.walk(WORKING_DIR):
        level = root.replace(str(WORKING_DIR), "").count(os.sep)
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")
        sub_indent = "  " * (level + 1)
        for f in files:
            print(f"{sub_indent}{f}")

screening_output/
  run_metadata.json
  output/
    final.parquet
    after_round_A.parquet
  round_A/
    agent_Screener/
      logs/
        item_A-5.jsonl
        item_A-2.jsonl
        item_A-0.jsonl
        item_A-7.jsonl
        item_A-9.jsonl
        item_A-8.jsonl
        item_A-6.jsonl
        item_A-3.jsonl
        item_A-4.jsonl
        item_A-1.jsonl
      memory/
      flags/
      results/
        item_A-5.json
        item_A-4.json
        item_A-8.json
        item_A-3.json
        item_A-7.json
        item_A-2.json
        item_A-9.json
        item_A-6.json
        item_A-1.json
        item_A-0.json


## Clean Up

In [11]:
if WORKING_DIR.exists():
    shutil.rmtree(WORKING_DIR)
    print(f"Removed {WORKING_DIR}")

Removed screening_output
